In [ ]:
#| default_exp callbacks.shared

# callbacks.shared

> YAML-injectable universal toolbox CBs.
> Auto-generated by nbdev from `nbs/api/callbacks/shared.ipynb`.

In [ ]:
#| export
from __future__ import annotations
from fastcore.all import *
import numpy as np
import pandas as pd
from typing import Optional, Any, Callable
from pydantic import BaseModel, Field
from marisco.callbacks.core import PerGroupCB, RemapCB

## Name normalisation

In [ ]:
#| export
class LowerStripNameCB(PerGroupCB):
    "Convert values to lowercase and strip any trailing spaces."
    def __init__(self, 
                 col_src: str, # Source column name e.g. 'Nuclide'
                 col_dst: str=None, # Destination column name
                 fn_transform: Callable=lambda x: x.lower().strip() # Transformation function
                 ):
        store_attr()
        self.__doc__ = f"Convert '{col_src}' column values to lowercase, strip spaces, and store in '{col_dst}' column."
        if not col_dst: self.col_dst = col_src
        
    def _safe_transform(self, value):
        "Ensure value is not NA and apply transformation function."
        return value if pd.isna(value) else self.fn_transform(str(value))

    def each_grp(self, grp, df, tfm): df[self.col_dst] = df[self.col_src].apply(self._safe_transform)

## Coordinate & uncertainty normalisation

In [ ]:
#| export
class SoftRelToAbsUncCB(PerGroupCB):
    "Convert relative uncertainty (%) to absolute; col_value=None is a safe Null-Object no-op."
    class Schema(BaseModel):
        col_value:   Optional[str] = None
        col_unc_rel: str           = "UNC_REL"
        factor:      float         = 100.0
    def __init__(self, col_value=None, col_unc_rel="UNC_REL", factor=100.0):
        self.cfg  = self.Schema(col_value=col_value, col_unc_rel=col_unc_rel, factor=factor)
        self.grps = [None] if col_value is None else None
    def each_grp(self, grp, df, tfm):                       # ZERO ast.If
        df["UNC"] = df[self.cfg.col_unc_rel] * df[self.cfg.col_value] / self.cfg.factor

In [ ]:
#| export
class SoftExtractUnitFromColCB(PerGroupCB):
    "Extract unit string from a column via regex; src_col=None is a safe Null-Object no-op."
    class Schema(BaseModel):
        src_col: Optional[str] = None
        dst_col: str           = "UNIT"
        pattern: str           = r"\((.*?)\)"             # default: parentheses
    def __init__(self, src_col=None, dst_col="UNIT", pattern=r"\((.*?)\)"):
        self.cfg  = self.Schema(src_col=src_col, dst_col=dst_col, pattern=pattern)
        self.grps = [None] if src_col is None else None
    def each_grp(self, grp, df, tfm):                       # ZERO ast.If
        df[self.cfg.dst_col] = df[self.cfg.src_col].str.extract(self.cfg.pattern, expand=False)

In [ ]:
#| export
class SoftShiftLonCB(PerGroupCB):
    "Shift longitude convention; shift=None is a safe Null-Object no-op."
    class Schema(BaseModel):
        col:   str            = "LON"
        shift: Optional[float] = None
    def __init__(self, col="LON", shift=None):
        self.cfg  = self.Schema(col=col, shift=shift)
        self.grps = [None] if shift is None else None
    def each_grp(self, grp, df, tfm):                       # ZERO ast.If
        df[self.cfg.col] = df[self.cfg.col] - self.cfg.shift

In [ ]:
#| export
class SoftDMStoDecimalCB(PerGroupCB):
    "Convert degree-minute-second columns to decimal degrees; col_deg=None is a safe Null-Object no-op."
    class Schema(BaseModel):
        col_deg: Optional[str] = None
        col_min: str           = "MIN"
        col_sec: str           = "SEC"
        col_dir: Optional[str] = None
        dst_col: str           = "LAT"
        neg_dir: list          = Field(default_factory=lambda: ["S", "W"])
    def __init__(self, col_deg=None, col_min="MIN", col_sec="SEC",
                 col_dir=None, dst_col="LAT", neg_dir=None):
        self.cfg  = self.Schema(col_deg=col_deg, col_min=col_min, col_sec=col_sec,
                                col_dir=col_dir, dst_col=dst_col,
                                neg_dir=neg_dir or ["S", "W"])
        self.grps = [None] if col_deg is None else None
    def each_grp(self, grp, df, tfm):                       # ZERO ast.If
        decimal = df[self.cfg.col_deg] + df[self.cfg.col_min] / 60 + df[self.cfg.col_sec] / 3600
        df[self.cfg.dst_col] = np.where(df[self.cfg.col_dir].isin(self.cfg.neg_dir),
                                        -decimal, decimal)

## Asset-based remapping

In [ ]:
#| export
class SoftAssetRemapCB(RemapCB):
    """YAML-injectable RemapCB; resolves LUT at construction time from a literal dict (lut),
    standard MARIS Excel LUT (lut_key), or a pre-resolved CSV file (asset_path)."""
    def __init__(self,
                 col_src:     str,
                 col_remap:   str,
                 lut:         dict  = None,   # Literal dict
                 lut_key:     str   = None,   # NC_DTYPES key -> get_lut(lut_key, key_col, val_col)
                 key_col:     str   = None,   # Key column for lut_key or asset_path
                 val_col:     str   = None,   # Value column for lut_key or asset_path
                 asset_path:  str   = None,   # Relative to lut_path(); loaded at construction time
                 default_val: int   = 0,
                 grps:        list  = None,
                ):
        from marisco.configs import get_lut, lut_path
        resolved = (lut                                               if lut is not None
                    else get_lut(lut_key, key=key_col, value=val_col) if lut_key is not None
                    else pd.read_csv(lut_path() / asset_path).set_index(key_col)[val_col].to_dict())
        super().__init__(lut=resolved, col_remap=col_remap, col_src=col_src,
                         default_val=default_val, grps=grps)

## Regex extraction

In [ ]:
#| export
class SoftRegexTransformCB(PerGroupCB):
    """Extract N named groups from one column via regex; src_col=None is a safe Null-Object no-op.
    Uses Pandas str.extract (vectorised). ZERO ast.If. Pydantic Schema.
    Fail-Fast: errors='raise' on non-numeric cast (Brake Rule 6)."""
    class Schema(BaseModel):
        src_col:  Optional[str]  = None
        pattern:  str            = r"(?P<VAL>.*)"
        dst_cols: list[str]      = Field(default_factory=list)
        cast:     str            = "str"   # "float", "int", or "str"
    def __init__(self,
                 src_col:  str   = None,   # Source column; None -> Null-Object no-op
                 pattern:  str   = r"(?P<VAL>.*)",  # Named-group regex
                 dst_cols: list  = None,   # Destination columns (must match group names)
                 cast:     str   = "str",  # Type cast for extracted values
                 grps:     list  = None,
                ):
        self.cfg  = self.Schema(src_col=src_col, pattern=pattern,
                                dst_cols=dst_cols or [], cast=cast)
        self.grps = [None] if src_col is None else grps
    def each_grp(self, grp, df, tfm):            # ZERO ast.If
        extracted = df[self.cfg.src_col].str.extract(self.cfg.pattern)
        cast_fn = {"float": pd.to_numeric, "int": lambda s: pd.to_numeric(s).astype(int),
                   "str": lambda s: s}.get(self.cfg.cast, lambda s: s)
        for col in self.cfg.dst_cols:
            df[col] = cast_fn(extracted[col])